# [재실행 전용 · 2026-10-04 회계 수정] kospi_rerun_colab.ipynb §6–§7

`ppo_runs.csv`(43.6%±3.7 / 샤프 1.099)는 회계 수정 **전** 환경으로 학습·평가한 값이다. §8에서 같은 시드를 수정된 환경으로 재학습하자 샤프가 1.157→1.134(iTransformer), 1.008→1.168(MTGNN)로 달라졌다.
이 노트북은 §1–§5(재클러스터링·백본·임베딩)는 다시 하지 않고 Drive에 있는 결과를 그대로 쓴 채, §6 PPO(백본 2종 × 시드 5개)만 수정된 환경으로 **재학습**하고 §7 요약을 다시 만든다.

- 기존 `ppo_runs.csv`·`ppo_summary.csv`는 `_pre_accounting_fix_20261004/`로 **이동**한다(§6은 완료된 시드를 건너뛰므로 남아 있으면 재학습이 안 됨).
- CPU 기준 PPO 1회 약 10분 → 10회 약 1.5–2시간. 세션이 끊기면 위에서부터 다시 실행하면 완료된 시드는 건너뛴다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q chronos-forecasting stable-baselines3[extra] gymnasium tslearn h5py pyarrow

## 0. 설정

In [ ]:
import gc, json, warnings
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
PARQUET    = DATA_DIR / 'kospi_valid.parquet'
OLD_CLUSTER_CSV = DATA_DIR / 'cluster_assignments.csv'
OUT_DIR    = DATA_DIR / 'kospi_rerun'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# sequences.h5(백본 학습 데이터)와 동일한 20개 피처·순서 — 학습과 추출 모두 이 리스트만 사용
FEATURE_COLS = ['Adj_Close', 'Open', 'High', 'Low', 'Volume',
                'SMA_5', 'SMA_20', 'SMA_60', 'EMA_12', 'EMA_26',
                'MACD', 'MACD_signal', 'MACD_hist', 'RSI_14', 'BB_width',
                'Volume_ratio', 'Return_1d', 'Return_5d', 'Volatility_20d', 'ATR_14']
N_FEATURES = len(FEATURE_COLS)
SEQ_LEN, PRED_LEN = 60, 5

# 백본: 목표 수익률(5일 뒤) 날짜 기준으로 분할 → RL 테스트(2024~)와 겹치지 않음
BB_TRAIN_END = '2023-06-30'
BB_VAL_END   = '2023-12-31'
MAX_TRAIN, MAX_VAL = 80_000, 30_000

# 클러스터링: RL 학습 구간까지만
CLUSTER_START, CLUSTER_END = '2021-01-01', '2023-12-31'
N_CLUSTERS, MIN_DAYS = 5, 700

# RL: rl_trading_colab.ipynb와 동일
RL_TRAIN_START, RL_TRAIN_END = '2021-01-01', '2023-12-31'
RL_TEST_START,  RL_TEST_END  = '2024-01-01', '2025-12-31'
TC = 0.001
PPO_TIMESTEPS = 300_000

BACKBONES = ['itransformer', 'mtgnn']
SEEDS = [42, 11, 22, 33, 44]

torch.manual_seed(42); np.random.seed(42)
print('설정 완료 →', OUT_DIR)

In [ ]:
raw = pd.read_parquet(PARQUET)
raw['Date'] = pd.to_datetime(raw['Date'])
raw['종목코드'] = raw['종목코드'].astype(str).str.zfill(6)
raw = raw.sort_values(['종목코드', 'Date']).reset_index(drop=True)
missing = [c for c in FEATURE_COLS + ['RetTarget_1d', 'Return_1d'] if c not in raw.columns]
assert not missing, f'parquet에 없는 컬럼: {missing}'
print(raw.shape, raw['Date'].min().date(), '~', raw['Date'].max().date(), '|', raw['종목코드'].nunique(), '종목')

# 종목별 배열로 한 번만 변환해 둔다
tickers = raw['종목코드'].unique().tolist()
per_ticker = {}
for tk, g in raw.groupby('종목코드', sort=False):
    per_ticker[tk] = {
        'feat': g[FEATURE_COLS].to_numpy(np.float32),
        'close': g['Adj_Close'].to_numpy(np.float64),
        'dates': g['Date'].dt.strftime('%Y-%m-%d').to_numpy(),
    }
print('종목 배열 준비 완료:', len(per_ticker))

def revin(w):
    # 윈도우별 정규화 — preprocess_sequences의 RevIN과 동일(nanmean/nanstd + 1e-8)
    mu = np.nanmean(w, axis=0, keepdims=True)
    sd = np.nanstd(w, axis=0, keepdims=True) + 1e-8
    return ((w - mu) / sd).astype(np.float32)

In [ ]:
# 수정 전 ppo_runs.csv / ppo_summary.csv를 백업 폴더로 이동 (이미 이동했으면 건너뜀)
import shutil
bk = OUT_DIR / '_pre_accounting_fix_20261004'
bk.mkdir(exist_ok=True)
for fn in ['ppo_runs.csv', 'ppo_summary.csv']:
    p = OUT_DIR / fn
    if p.exists() and not (bk / fn).exists():
        shutil.move(str(p), str(bk / fn))
        print('이동:', fn)
    elif p.exists():
        print(f'{fn}: 백업본이 이미 있음 — 현재 파일은 이번 재실행 결과로 보고 유지(이어서 실행)')
print('백업 폴더:', bk, sorted(x.name for x in bk.iterdir()))


## 6. PPO 실행 — 백본 2종 × 시드 5개

환경·보상·하이퍼파라미터는 `rl_trading_colab.ipynb`의 `KospiPortfolioEnv`와 같다. 속도를 위해 날짜별 클러스터 평균 임베딩과 클러스터 수익률을 미리 계산해 두는 점만 다르다(계산 결과는 같다).

**이어서 실행**: 런타임이 끊기면 §0(설정·데이터 로드 셀) → 아래 셀을 다시 실행한다. 이미 `ppo_runs.csv`에 있는 (백본, 시드)는 건너뛴다.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

clusters = pd.read_csv(OUT_DIR / 'cluster_assignments_2021_2023.csv', dtype={'종목코드': str})
clusters['종목코드'] = clusters['종목코드'].str.zfill(6)
cl_of = dict(zip(clusters['종목코드'], clusters['cluster']))

ret_df = raw[['종목코드', 'Date', 'RetTarget_1d']].dropna()
ret_df = ret_df[ret_df['종목코드'].isin(cl_of)]
ret_df['c'] = ret_df['종목코드'].map(cl_of)
ret_df['d'] = ret_df['Date'].dt.strftime('%Y-%m-%d')
cl_ret = ret_df.groupby(['d', 'c'])['RetTarget_1d'].mean().unstack().reindex(columns=range(N_CLUSTERS)).fillna(0.0)

def load_cluster_embs(name):
    with h5py.File(OUT_DIR / f'rl_embeddings_{name}_fixed.h5', 'r') as f:
        tk = [x.decode() for x in f['tickers'][:]]; dt = [x.decode() for x in f['dates'][:]]
        E = f['embeddings'][:]
    df = pd.DataFrame({'tk': tk, 'd': dt}); df['c'] = df['tk'].map(cl_of)
    df = df.dropna(subset=['c']); idx = df.index.to_numpy()
    out = {}
    for (d, c), g in df.groupby(['d', 'c']).groups.items():
        out[(d, int(c))] = E[g].mean(0)
    dates = sorted({d for d, _ in out})
    M = np.zeros((len(dates), N_CLUSTERS, 64), np.float32)
    for i, d in enumerate(dates):
        for c in range(N_CLUSTERS):
            if (d, c) in out: M[i, c] = out[(d, c)]
    return dates, M

class KospiPortfolioEnv(gym.Env):
    # rl_trading_colab.ipynb와 동일한 동역학: 상태=클러스터 평균 임베딩+현재 비중, 행동=softmax 비중,
    # 보상=다음 날짜 클러스터 등가중 RetTarget_1d 수익률 − 거래비용
    def __init__(self, dates, M_by_date, cl_ret, tc=TC):
        super().__init__()
        self.dates, self.M, self.R, self.tc = dates, M_by_date, cl_ret, tc
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(N_CLUSTERS * 64 + N_CLUSTERS,), dtype=np.float32)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(N_CLUSTERS,), dtype=np.float32)
    def _obs(self): return np.concatenate([self.M[self.dates[self.t]].ravel(), self.w]).astype(np.float32)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); self.t, self.w, self.v = 0, np.ones(N_CLUSTERS, np.float32) / N_CLUSTERS, 1.0
        return self._obs(), {}
    def step(self, a):
        a = np.exp(a - a.max()); a = a / a.sum()
        cost = self.tc * np.abs(a - self.w).sum()
        nd = self.dates[min(self.t + 1, len(self.dates) - 1)]
        rv = self.R.get(nd, np.zeros(N_CLUSTERS))
        r = float((a * rv).sum())
        # [2026-10-04 회계 수정] 다음 스텝 비용은 가격 변동 후 실제 보유 비중 대비로 계산
        g = 1.0 + r
        self.v *= (1 + r - cost); self.w = (a * (1 + rv) / g if g > 1e-12 else a).astype(np.float32); self.t += 1
        return self._obs(), float(r - cost), self.t >= len(self.dates) - 1, False, {'portfolio_value': self.v}

def perf(vals):
    r = np.diff(vals) / vals[:-1]
    return {'total_ret': vals[-1] / vals[0] - 1, 'sharpe': r.mean() / (r.std() + 1e-8) * np.sqrt(252),
            'mdd': float(((vals / np.maximum.accumulate(vals)) - 1).min())}

def backtest(env, policy=None):
    obs, _ = env.reset(); done, vals = False, [1.0]
    eq = np.ones(N_CLUSTERS, np.float32) / N_CLUSTERS
    while not done:
        a = policy.predict(obs, deterministic=True)[0] if policy else np.log(eq)
        obs, _, done, _, info = env.step(a); vals.append(info['portfolio_value'])
    return np.array(vals)
print('환경 정의 완료')

In [ ]:
%%time
runs_csv = OUT_DIR / 'ppo_runs.csv'
done_runs = pd.read_csv(runs_csv) if runs_csv.exists() else pd.DataFrame(columns=['backbone', 'seed'])

for name in BACKBONES:
    dates, M = load_cluster_embs(name)
    M_by_date = {d: M[i] for i, d in enumerate(dates)}
    R = {d: cl_ret.loc[d].to_numpy(np.float32) for d in cl_ret.index}
    tr_dates = [d for d in dates if RL_TRAIN_START <= d <= RL_TRAIN_END]
    te_dates = [d for d in dates if RL_TEST_START <= d <= RL_TEST_END]
    ew = perf(backtest(KospiPortfolioEnv(te_dates, M_by_date, R)))
    print(f'[{name}] 학습 {len(tr_dates)}일 / 테스트 {len(te_dates)}일 | 등가중 테스트 {ew}')
    for seed in SEEDS:
        if ((done_runs['backbone'] == name) & (done_runs['seed'] == seed)).any():
            print(f'  seed={seed} 이미 완료 — 건너뜀'); continue
        ppo = PPO('MlpPolicy', KospiPortfolioEnv(tr_dates, M_by_date, R), learning_rate=3e-4, n_steps=512,
                  batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
                  vf_coef=0.5, max_grad_norm=0.5, policy_kwargs=dict(net_arch=[256, 256]),
                  verbose=0, device='cpu', seed=seed)
        ppo.learn(total_timesteps=PPO_TIMESTEPS)
        res = perf(backtest(KospiPortfolioEnv(te_dates, M_by_date, R), ppo))
        row = {'backbone': name, 'seed': seed, **{f'ppo_{k}': v for k, v in res.items()},
               **{f'ew_{k}': v for k, v in ew.items()}}
        done_runs = pd.concat([done_runs, pd.DataFrame([row])], ignore_index=True)
        done_runs.to_csv(runs_csv, index=False)
        print(f'  seed={seed}: PPO {res["total_ret"]:+.2%} / 샤프 {res["sharpe"]:.3f} | 등가중 {ew["total_ret"]:+.2%} / {ew["sharpe"]:.3f}')

## 7. 요약과 판정

In [ ]:
runs = pd.read_csv(OUT_DIR / 'ppo_runs.csv')
runs['beats_ew'] = runs['ppo_sharpe'] > runs['ew_sharpe']
summary = runs.groupby('backbone').agg(
    n=('seed', 'count'), ret_mean=('ppo_total_ret', 'mean'), ret_std=('ppo_total_ret', 'std'),
    sharpe_mean=('ppo_sharpe', 'mean'), sharpe_std=('ppo_sharpe', 'std'),
    beats_ew=('beats_ew', 'sum'), ew_ret=('ew_total_ret', 'first'), ew_sharpe=('ew_sharpe', 'first'))
print(runs.round(4).to_string(index=False)); print(); print(summary.round(4).to_string())
summary.to_csv(OUT_DIR / 'ppo_summary.csv', encoding='utf-8-sig')
print('\n참고(원래 파이프라인, 누수·버그 포함, 시드 42 1회): iTransformer +137.22% / 2.473, MTGNN +0.28% / 0.100, 등가중 +34.33% / 1.043')

### 판정 기준 (논문 반영 방법)

- **두 백본 모두 대부분의 시드에서 등가중보다 샤프가 낮다** → 1.1절의 "KOSPI 단일 정책 실패" 유지. 수치를 이 노트북 값(평균±표준편차)으로 교체하고, 원래 +0.28%는 버그·누수가 있던 파이프라인 값이라 각주로 내린다.
- **결과가 백본에 따라 갈리거나 시드별로 엇갈린다** → 1.1절을 "KOSPI 결과는 백본·시드에 민감했다"로 바꾸고, 단일 정책 실패의 근거는 크립토(−75.68%)와 미국 confound 실험으로 옮긴다. 1.4절 기여 #1과 7장 종합표도 같은 방향으로 수정한다.
- **두 백본 모두 대부분 등가중을 이긴다** → "KOSPI 단일 정책 실패" 주장은 철회. 원래 실패는 누수·버그의 산물이었다고 명시한다(한계 절).

결과 CSV(`ppo_runs.csv`, `ppo_summary.csv`, `backbone_val_metrics.json`)를 Claude에게 넘기면 논문 수정안을 만든다.